# Adobe Leadership Agent Demo

I use this notebook as the fastest review path for the assignment.

What I can do from here:

1. Set `OPENAI_API_KEY`, model names, and parser choice in one place.
2. Rebuild ingestion and indexes if I want a fresh run.
3. Ask a single natural-language leadership question without using the CLI.
4. Show the markdown answer directly in the notebook and save the JSON report under `outputs/sample_answers/`.

Notes:

- The preview section below shows the latest saved checked answers.
- If I run the last cell, it generates a fresh answer, so wording can differ slightly from the preview.
- If I want the fresh run to match the preview as closely as possible, I should keep `PARSER_PRIMARY` and `PARSER_FALLBACK` both set to `docling` and use deterministic mode by leaving `OPENAI_API_KEY` blank.


## Latest Checked Outputs

I am showing the latest saved answers here so the main outcomes are visible immediately when this notebook opens.

These lines are copied from the current JSON reports in `outputs/sample_answers/` using the `docling` parser path.

- `What is our current revenue trend?`  
  `Revenue shows an upward trend from Q1FY24 (5,182) to Q2FY25 (5,873), a change of 13.3%.`

- `Which departments are underperforming?`  
  `Digital Experience appears to be underperforming relative to Digital Media, with 4.3% change from Q4FY24 to Q2FY25 versus 4.8%.`

- `What were the key risks highlighted in the last quarter?`  
  `In the most recent quarter materials, the key risks highlighted were primarily forward-looking business and operational risks: failure to innovate and meet customer needs; issues related to AI development and use; failure to compete effectively; reputational/brand damage; inability to realize benefits from investments or acquisitions; IT service interruptions or failures (including third-party systems); security incidents; weak third-party business relationships; adverse macroeconomic conditions and multinational exposure; and complex sales cycles. The materials also flagged additional risks such as recruiting/retaining key personnel, litigation/regulatory/IP claims, compliance with global laws and privacy/security rules, foreign exchange fluctuations, revenue-recognition timing, debt obligations, and stock-price volatility.`

The corresponding files are:

- `outputs/sample_answers/docling_what_is_our_current_revenue_trend.json`
- `outputs/sample_answers/docling_which_departments_are_underperforming.json`
- `outputs/sample_answers/docling_what_were_the_key_risks_highlighted_in_the_last_quarter.json`

Validation notes:

- These files were regenerated from a fresh `docling` parse with FAISS-backed indexes.
- OpenAI was used for classification/synthesis, while computed quantitative answers are kept deterministic when they are stronger than narrative paraphrasing.
- I also checked the same three questions across other parser paths earlier in the workflow.


In [5]:
from pathlib import Path
import json

from IPython.display import Markdown, display

from leadership_agent.answering import LeadershipAgent, render_report, save_report
from leadership_agent.config import AppConfig
from leadership_agent.utils import slugify

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "config.yaml"

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


Project root: d:\Preetam\preets\adobe\adobe_assignment
Config path: d:\Preetam\preets\adobe\adobe_assignment\config.yaml


In [ ]:
# Edit these values if you want to test with your own key or model choices.
# Leave OPENAI_API_KEY blank to use the deterministic offline path.

OPENAI_API_KEY = ""
OPENAI_NANO_MODEL = "gpt-5.4-nano"
OPENAI_MINI_MODEL = "gpt-5.4-mini"
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

PARSER_PRIMARY = "docling"
PARSER_FALLBACK = "docling"

FORCE_REBUILD = True
QUESTION = "What is the trend in Digital Media ARR over the last few quarters?"
REPORT_FILENAME = None  # Example: "review_answer.json"

SAMPLE_QUESTIONS = [
    "What is our current revenue trend?",
    "Which departments are underperforming?",
    "What were the key risks highlighted in the last quarter?",
]

SAMPLE_QUESTIONS


['What is our current revenue trend?',
 'Which departments are underperforming?',
 'What were the key risks highlighted in the last quarter?']

In [7]:
def build_notebook_config() -> AppConfig:
    config = AppConfig.load(CONFIG_PATH)
    config.models.nano = OPENAI_NANO_MODEL.strip() or config.models.nano
    config.models.mini = OPENAI_MINI_MODEL.strip() or config.models.mini
    config.models.embedding = EMBEDDING_MODEL.strip() or config.models.embedding
    config.parser.primary = PARSER_PRIMARY.strip() or config.parser.primary
    config.parser.fallback = PARSER_FALLBACK.strip() or config.parser.fallback
    if OPENAI_API_KEY.strip():
        config.runtime.llm_provider = "openai"
        config.runtime.openai_api_key = OPENAI_API_KEY.strip()
    else:
        config.runtime.llm_provider = "extractive"
        config.runtime.openai_api_key = ""
    return config


def parsed_ready(config: AppConfig) -> bool:
    required = [
        config.parsed_data_dir / "sections" / "sections.jsonl",
        config.parsed_data_dir / "chunks" / "chunks.jsonl",
        config.parsed_data_dir / "tables" / "tables.jsonl",
    ]
    return all(path.exists() for path in required)


def indexes_ready(config: AppConfig) -> bool:
    required = [
        config.index_dir / "bm25" / "sections.pkl",
        config.index_dir / "bm25" / "chunks.pkl",
        config.index_dir / "dense" / "sections.meta.json",
        config.index_dir / "metadata" / "leadership.duckdb",
    ]
    return all(path.exists() for path in required)


def prepare_pipeline(config: AppConfig, force_rebuild: bool = False):
    agent = LeadershipAgent(config)
    status = {}
    if force_rebuild or not parsed_ready(config):
        status["ingest"] = agent.ingest()
    else:
        status["ingest"] = "skipped"
    if force_rebuild or not indexes_ready(config):
        status["build_index"] = agent.build_index()
    else:
        status["build_index"] = "skipped"
    return agent, status


def ask_notebook_question(question: str, report_filename: str | None = None, force_rebuild: bool = False):
    config = build_notebook_config()
    agent, prep_status = prepare_pipeline(config, force_rebuild=force_rebuild)
    report = agent.ask(question, output_dir=config.output_dir)

    filename = (report_filename or f"{slugify(question)}.json").strip()
    if not filename.lower().endswith(".json"):
        filename += ".json"
    report_path = config.output_dir / "sample_answers" / filename
    save_report(report, report_path)

    summary = {
        "question": question,
        "llm_provider": config.runtime.llm_provider,
        "nano_model": config.models.nano,
        "mini_model": config.models.mini,
        "embedding_model": config.models.embedding,
        "parser_primary": config.parser.primary,
        "parser_fallback": config.parser.fallback,
        "report_path": str(report_path),
    }

    print("Run summary:")
    print(json.dumps(summary, indent=2))
    print("\nPreparation status:")
    print(json.dumps(prep_status, indent=2))
    print("\nRendered answer:\n")
    display(Markdown(render_report(report)))
    return report, report_path, prep_status


In [8]:
report, report_path, prep_status = ask_notebook_question(
    question=QUESTION,
    report_filename=REPORT_FILENAME,
    force_rebuild=FORCE_REBUILD,
)

report_path


2026-03-23 18:26:54,804 INFO docling.datamodel.document | detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-23 18:26:54,842 INFO docling.document_converter | Going to convert document batch...
2026-03-23 18:26:54,842 INFO docling.document_converter | Initializing pipeline for StandardPdfPipeline with options hash a6a2eef09bfdbd6bdaeaf15fcaebde59
2026-03-23 18:26:54,842 INFO docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
[INFO] 2026-03-23 18:26:54,882 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-23 18:26:54,890 [RapidOCR] download_file.py:60: File exists and is valid: D:\Preetam\preets\adobe\adobe_assignment\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-23 18:26:54,890 [RapidOCR] main.py:53: Using D:\Preetam\preets\adobe\adobe_assignment\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-23 18:26:55,168 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-23 18:2

Run summary:
{
  "question": "What is the trend in Digital Media ARR over the last few quarters?",
  "llm_provider": "openai",
  "nano_model": "gpt-5.4-nano",
  "mini_model": "gpt-5.4-mini",
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "parser_primary": "docling",
  "parser_fallback": "docling",
  "report_path": "outputs\\sample_answers\\what_is_the_trend_in_digital_media_arr_over_the_last_few_quarters.json"
}

Preparation status:
{
  "ingest": {
    "documents": 3,
    "sections": 322,
    "chunks": 347,
    "tables": 84,
    "source_dir": "data\\raw",
    "parser_chain": [
      "docling",
      "native"
    ]
  },
  "build_index": {
    "documents": 3,
    "sections": 322,
    "chunks": 347,
    "tables": 84,
    "dense": {
      "sections": {
        "backend": "fastembed",
        "model": "BAAI/bge-small-en-v1.5",
        "count": 322,
        "usage": {
          "model": "BAAI/bge-small-en-v1.5",
          "errors": []
        }
      },
      "chunks": {
        "backend":

## Direct Answer
Digital Media ARR has been trending upward over the last few quarters. The latest disclosed figure is $18.09 billion exiting Q2 FY2025, up 12.1% year over year. The annual report shows Digital Media ARR at $17.33 billion exiting FY2024, up from $15.33 billion at FY2023, indicating continued growth. However, the provided evidence does not include a full quarter-by-quarter ARR series for the intervening quarters, so the precise quarter-over-quarter path cannot be fully reconstructed from the supplied materials.

## Key Evidence
- Digital Media ARR exiting Q2 FY2025 was $18.09 billion, representing 12.1% year-over-year growth.
- Total Digital Media ARR grew to $17.33 billion at the end of fiscal 2024, up from $15.33 billion at the end of fiscal 2023.
- The investor datasheet shows Digital Media revenue rising each quarter from Q1 FY2023 through Q2 FY2025, which is consistent with an upward business trend, though this is revenue rather than ARR.

## Sources
- adobe_q2_fy2025_earnings_release.pdf | Second Quarter Fiscal Year 2025 Business Segment Highlights | pages n/a
- adobe_fy2024_annual_report.pdf | | Digital Media ARR | Creative ARR + Document Cloud ARR | | pages n/a
- adobe_q2_fy2025_investor_datasheet.pdf | Document Overview | pages n/a

## Caveats
- A complete quarter-by-quarter Digital Media ARR series was not provided in the evidence.
- The datasheet includes quarterly Digital Media revenue, not quarterly ARR, so it can only support a directional inference, not exact ARR quarter-over-quarter changes.
- ARR values are affected by currency revaluation, as noted in the annual report, which can make comparisons across periods less directly comparable.
- The provided evidence does not include a quarter-by-quarter Digital Media ARR series; it only provides ARR for exiting Q2 FY2025 and total Digital Media ARR for FY2024 and FY2023.
- The cited investor datasheet evidence (id 1 and id 9) contains Digital Media revenue by quarter, not Digital Media ARR by quarter, so it cannot substantiate an ARR trend across intervening quarters.
- The answer claims an upward trend “over the last few quarters,” but the evidence only supports directionally for Q2 FY2025 vs prior fiscal year end (FY2024) and does not show the intermediate quarter path.

WindowsPath('outputs/sample_answers/what_is_the_trend_in_digital_media_arr_over_the_last_few_quarters.json')